In [84]:
import re
import os
import json
import time
import requests
import pandas as pd
from IPython.display import display

MAPBOX_TOKEN = 'pk.eyJ1IjoiaGJ1c2huZWxsIiwiYSI6ImNtbTNrYno0ZDAxb2UycXF6dTUwM3VyMmcifQ.jivZrXJ2wj5ml73B4BpPwg'
BATCH_URL = f"https://api.mapbox.com/search/geocode/v6/batch?access_token={MAPBOX_TOKEN}"

CHECKPOINT_FILE = 'geocoded_irish_properties.csv'
BATCH_SIZE = 1000

In [85]:
def prep_irish_address_for_mapbox(row, address_col='Address', county_col='County'):
    if pd.isna(row[address_col]):
        return None
        
    address = str(row[address_col]).upper().strip()
    county = str(row[county_col]).upper().strip()

    # 1. Strip internal unit prefixes (APT, FLAT, UNIT, NO, NOS) and their numbers
    # because mapbox goes by building
    address = re.sub(r'^(?:APT|FLAT|UNIT|NO|NOS)\.?\s*\w+\s*-?\s*', '', address)
    
    # 2. Collapse explicit ranges (1-5, 1 TO 5, 1/2, 1 & 2) -> Keeps the first number
    # becayse mapbox only wants one number
    address = re.sub(r'\b(\d+[A-Z]*)\s*(?:-|TO|/|&|AND)\s*\d+[A-Z]*\b', r'\1', address)
    
    # 3. Collapse space-separated lists at the start (1 2 3 Corrib View -> 1 Corrib View)
    address = re.sub(r'^(\d+[A-Z]*)\s+\d+(?:\s+\d+)*\s+', r'\1 ', address)

    # 5. Clean up
    address = re.sub(r'\s{2,}', ' ', address)
    address = re.sub(r'^,\s*', '', address).strip()
    
    county = f"CO. {county}"

    # having county include CO. prefix will 
    if not address.endswith(county):
        address = f"{address}, {county}"
        
    return address

In [86]:
raw_df = pd.read_csv('../data/ppr-group-25200353-train-cleaned.csv')

print(f"loaded {len(raw_df)} records")

raw_df['Mapbox Query'] = raw_df.apply(
    prep_irish_address_for_mapbox,
    axis=1, 
    address_col='Address',
    county_col='County'
)

display(raw_df[['Address', 'Mapbox Query']].head())

loaded 53992 records


,Address,Mapbox Query
0,"APT 14, RUSSELL COURT, FELTRIM RD","RUSSELL COURT, FELTRIM RD, CO. DUBLIN"
1,"19 PARK VILLAS, DEMESNE RD, DUNDALK","19 PARK VILLAS, DEMESNE RD, DUNDALK, CO. LOUTH"
2,"No 2 The Grove, Cahereen Heights, Castleisland","THE GROVE, CAHEREEN HEIGHTS, CASTLEISLAND, CO...."
3,"APT 4 BLOCK 6, WOODFORD, WHEATON HALL","BLOCK 6, WOODFORD, WHEATON HALL, CO. LOUTH"
4,"18 GERALDINE ST, DUBLIN 7, DUBLIN","18 GERALDINE ST, DUBLIN 7, DUBLIN, CO. DUBLIN"


In [87]:
if os.path.exists(CHECKPOINT_FILE):
    df = pd.read_csv(CHECKPOINT_FILE)
    processed_count = df['Confidence'].notna().sum()
    print(f"loaded checkpoint. {processed_count} addresses already processed")
else:
    df = raw_df.copy()
    df['Latitude'] = pd.NA
    df['Longitude'] = pd.NA
    df['Confidence'] = pd.NA
    
    df.to_csv(CHECKPOINT_FILE, index=False)
    print("created new checkpoint file")

created new checkpoint file


In [102]:
def run_geocoding_loop(df):
    while True:
        unprocessed = df[df['Confidence'].isna()]
        
        if unprocessed.empty:
            print("all addresses processed")
            break
            
        batch_df = unprocessed.head(BATCH_SIZE)
        indices = batch_df.index.tolist()
        
        payload = [{"q": str(query), "country": "ie", "limit": 1} for query in batch_df['Mapbox Query']]
        print(f"sending {BATCH_SIZE} addresses...")
        
        response = requests.post(BATCH_URL, json=payload)
        if response.status_code != 200:
            print(f"API Error {response.status_code}: {response.text}")
            break
            
        data = response.json()
        results_array = data.get('batch', data) if isinstance(data, dict) else data
        needs_review = False
        
        for i, res in enumerate(results_array):
            df_idx = indices[i]
            features = res.get('features', [])
            
            if features:
                feature = features[0]
                coords = feature['geometry']['coordinates']
                props = feature['properties']
                
                feature_type = props.get('feature_type', 'unknown')
                match_code_dict = props.get('match_code', {})
                
                conf = props.get('confidence')
                if not conf:
                    conf = match_code_dict.get('confidence', 'none')
                
                df.loc[df_idx, 'Longitude'] = coords[0]
                df.loc[df_idx, 'Latitude'] = coords[1]
                df.loc[df_idx, 'Feature_Type'] = feature_type
                df.loc[df_idx, 'Confidence'] = conf
                df.loc[df_idx, 'Match_Code'] = json.dumps(match_code_dict)
                
            else:
                df.loc[df_idx, 'Confidence'] = 'not_found'
                df.loc[df_idx, 'Feature_Type'] = 'not_found'
                needs_review = True
                
        df.to_csv(CHECKPOINT_FILE, index=False)
        
        if needs_review:
            print("mapbox cannot find address. review in next cell")
            break
        else:
            print("moving to next...")

run_geocoding_loop(df)

sending 1000 addresses...
moving to next...
sending 1000 addresses...
moving to next...
sending 1000 addresses...
moving to next...
all addresses processed


In [104]:
from IPython.display import clear_output

failed_mask = df['Confidence'] == 'not_found'
failed_indices = df[failed_mask].index

if len(failed_indices) == 0:
    print("no failed addresses to review")
else:
    print(f"found {len(failed_indices)} addresses that need manual entry\n")
    
    for idx in failed_indices:
        raw_address = df.loc[idx, 'Address']
        query = df.loc[idx, 'Mapbox Query']
        
        print(f"Row {idx} | Raw Address: {raw_address}")
        print(f"Mapbox Query: {query}")
        
        lat_input = input("Enter Latitude (or press Enter to skip & leave blank): ").strip()
            
        lon_input = input("Enter Longitude: ").strip()
        
        try:
            df.loc[idx, 'Latitude'] = float(lat_input)
            df.loc[idx, 'Longitude'] = float(lon_input)
            df.loc[idx, 'Confidence'] = 'manual'
            df.loc[idx, 'Feature_Type'] = 'manual'
            df.loc[idx, 'Match_Code'] = '{"manual_override": true}'
            
            clear_output(wait=True)
            print(f"saved manual coordinates for row {idx}\n")
            
        except ValueError:
            clear_output(wait=True)
            print(f"invalid coordinate format. Skipped row {idx}\n")

    # Save your manual work immediately
    df.to_csv(CHECKPOINT_FILE, index=False)
    print("row saved")

no failed addresses to review
